In [86]:
import os, re, json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
solar_api = os.getenv("SOLAR_API")

client = OpenAI(
    api_key=solar_api,
    base_url="https://api.upstage.ai/v1"
)


In [87]:
content = """

2. 주요 제품 및 서비스

                                                            (단위 : 백만원)

매출유형

품 목

제23기

제22기

제21기

제20기

제품

큐피스템

줄기세포배양액 등

5,934	1,200	699	
926

상품

레모둘린 등

657	5,832	
3,451

3,218

기타(용역/기술 매출 등)


1,089	-	
3

합    계

6,591	8,121	
4,151

4,147










"""

In [88]:
prompt = """
기업 사업보고서의 주요 제품/서비스 표에서
기간별 매출 금액을 표 형태 그대로 행렬로 추출한다.

비율 계산 절대 하지 마라.

────────────────
[출력 형식 — 반드시 이 구조]
────────────────

{
  "기간": [기간1, 기간2, ...],
  "품목1": [금액1, 금액2, ...],
  "품목2": [금액1, 금액2, ...],
  ...
  "합계": [합계1, 합계2, ...]
}

────────────────
[규칙]
────────────────

1. 표 열 순서 그대로 유지
2. 모든 품목 배열 길이 = 기간 배열 길이
3. 공란 / '-' = 0
4. 숫자만 출력 (쉼표 제거)
5. 품목명 그대로 사용 (병합셀 분리 금지)
6. JSON만 출력
"""


In [89]:
import json, re
from IPython.display import display, JSON

# result → JSON
data = json.loads(re.search(r"\{.*\}", result, re.S).group())

# 최신기 index
def period_num(p):
    m = re.search(r"\d+", p)
    return int(m.group()) if m else -1

idx = max(range(len(data["기간"])), key=lambda i: period_num(data["기간"][i]))

latest_period = data["기간"][idx]

# 합계
total = data["합계"][idx]

# 비중 계산
ratio = {}
for k, v in data.items():
    if k in ("기간", "합계"):
        continue
    ratio[k] = f"{(v[idx]/total*100):.2f}%"

ratio["합계"] = "100.00%"

latest_only = {latest_period: ratio}

print(json.dumps(latest_only, indent=2, ensure_ascii=False))
display(JSON(latest_only))


KeyError: '기간'

In [ ]:

final_input = f"""
다음 문서에서 주요 제품별 매출 비중을 추출하라.

문서:
{content}

지침:
{prompt}
"""


In [ ]:
messages = [
    {"role": "user", "content": final_input}
]


In [ ]:

response = client.chat.completions.create(
    model="solar-1-mini-chat",   
    messages=messages,
    temperature=0
)

result = response.choices[0].message.content
print("✅ result 생성 완료")


✅ result 생성 완료


In [ ]:
print("===== LLM AMOUNT DATA =====")
print(json.dumps(amount_data, indent=2, ensure_ascii=False))


===== LLM AMOUNT DATA =====
{
  "제23기": {
    "큐피스템, 줄기세포배양액 등": 5934,
    "레모둘린 등": 657,
    "기타(용역/기술 매출 등)": 1089,
    "합계": 7679
  }
}


In [ ]:
import re, json
from IPython.display import display, JSON

m = re.search(r"\{.*\}", result, flags=re.S)
amount_data = json.loads(m.group())

def _period_num(k: str):
    mm = re.search(r"제\s*(\d+)\s*기", k)
    return int(mm.group(1)) if mm else -1

latest_period = max(amount_data.keys(), key=_period_num)
items = amount_data[latest_period]

# ⭐ 합계 처리 (핵심 수정)
total = items.get("합계")
if not total or total == 0:
    total = sum(v for k, v in items.items() if k != "합계")
    print(f"⚠ 합계 미검출 → 품목 합으로 계산: {total}")

# 비중 계산
ratio = {}
for k, v in items.items():
    if k == "합계":
        continue
    ratio[k] = f"{(v/total*100):.2f}%"

ratio["합계"] = "100.00%"

latest_only = {latest_period: ratio}

print("===== LATEST ONLY (CALCULATED) =====")
print(json.dumps(latest_only, indent=2, ensure_ascii=False))
display(JSON(latest_only))


===== LATEST ONLY (CALCULATED) =====
{
  "제23기": {
    "큐피스템, 줄기세포배양액 등": "77.28%",
    "레모둘린 등": "8.56%",
    "기타(용역/기술 매출 등)": "14.18%",
    "합계": "100.00%"
  }
}


<IPython.core.display.JSON object>

In [ ]:
#json파싱 검증
import json

try:
    data = json.loads(result)
except json.JSONDecodeError:
    print("⚠ JSON 파싱 실패 → 재요청 필요")

#합계 100 검증
def check_total(data):
    for period, items in data.items():
        if "합계" in items:
            total = float(items["합계"].replace("%",""))
            if abs(total - 100) > 0.1:
                print(f"⚠ 합계 이상: {period}")

#빈 결과 방지
if not data:
    print("⚠ 주요 제품 매출 비중 추출 실패")
